In [1]:
import os
from pathlib import Path

import pandas as pd
import polars as pl
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import scipy
import sklearn
import tensorflow as tf
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

import mllabs  # ml-labs: our experiment management framework

import sys
print(sys.version)

for i in [pd, pl, np, plt, sns, scipy, sklearn, tf, xgb, lgb, cb, mllabs]:
    if hasattr(i, '__version__'):
        print(i.__name__, i.__version__)
    else:
        print(i.__name__)

from IPython.display import Markdown

from mllabs import Connector, Experimenter, ColSelector
from mllabs.collector import MetricCollector, ModelAttrCollector, StackingCollector
from mllabs.adapter import XGBoostAdapter, LightGBMAdapter, CatBoostAdapter
from mllabs.nn import NNClassifier, FTTransformerHead
from mllabs.col import ohe_drop_first

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from mllabs.processor import PolarsLoader, PandasConverter, ExprProcessor
from sklearn.pipeline import make_pipeline

data_path = Path('data')

# Binary-encode several categorical features using Polars expressions.
# We also derive indicator columns for internet-service-related features
# ('No internet service' is treated as a separate binary flag).
dict_expr = {
    'gender': (pl.col('gender') == 'Male').cast(pl.Int8),
    'No_Internet': (pl.col('DeviceProtection') == 'No internet service').cast(pl.Int8),
    'DSL_Y': (pl.col('InternetService') == 'DSL').cast(pl.Int8)
}
for i in ['Dependents', 'PaperlessBilling', 'Partner', 'PhoneService']:
    dict_expr[i] = (pl.col(i) == 'Yes').cast(pl.Int8)

for i in ['DeviceProtection', 'OnlineBackup', 'OnlineSecurity', 'StreamingMovies', 'StreamingTV', 'TechSupport', 'MultipleLines']:
    dict_expr[i + '_Y'] = (pl.col(i) == 'Yes').cast(pl.Int8)

# PolarsLoader reads CSVs into Polars DataFrames.
# ExprProcessor applies the expression dict above in a single vectorized pass.
# PandasConverter converts to pandas, setting 'id' as the index.
loader = make_pipeline(
    PolarsLoader(predefined_types={'id': pl.Int64}),
    ExprProcessor(dict_expr=dict_expr)
)

df_train = loader.fit_transform([data_path / 'train.csv']).with_columns(
    (pl.col('Churn') == 'Yes').cast(pl.Int8)
)
df_org = loader.transform([data_path / 'WA_Fn-UseC_-Telco-Customer-Churn.csv']).drop('customerID').with_columns(
    id=-pl.int_range(1, pl.len() + 1),
    tenure=pl.col('tenure').clip(1),
    Churn=(pl.col('Churn') == 'Yes').cast(pl.Int8)
)
df_test = loader.transform([data_path / 'test.csv'])

# Variable groupings
# X_bin  : original binary features (Yes/No encoded as 0/1)
# X_tri  : 3-level categorical features (Yes / No / No internet service)
# X_bin2 : derived binary flags from X_tri (e.g. OnlineSecurity_Y)
# X_num  : numerical features
# X_nom  : nominal categoricals kept as strings for tree-based models
X_bin = ['Dependents', 'PaperlessBilling', 'Partner', 'PhoneService', 'gender', 'SeniorCitizen']
X_tri = ['DeviceProtection', 'OnlineBackup', 'OnlineSecurity', 'StreamingMovies', 'StreamingTV', 'TechSupport']
X_bin2 = ['{}_Y'.format(i) for i in X_tri] + ['No_Internet', 'DSL_Y', 'MultipleLines_Y']
X_tri.append('InternetService')
X_tri.append('MultipleLines')
X_num = ['TotalCharges', 'MonthlyCharges', 'tenure']
X_nom = ['PaymentMethod', 'Contract']
X_all = X_bin + X_tri + X_bin2 + X_num + X_nom

target = 'Churn'

2026-03-18 09:54:58.454861: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


3.12.6 (main, Sep 30 2024, 02:19:13) [GCC 9.4.0]
pandas 2.3.3
polars 1.39.0
numpy 2.1.3
matplotlib.pyplot
seaborn 0.13.2
scipy 1.15.2
sklearn 1.5.2
tensorflow 2.20.0
xgboost 3.2.0
lightgbm 4.6.0
catboost 1.2.8
mllabs 0.6.3


In [2]:
from sklearn.linear_model import LinearRegression
X_reg = ['PhoneService', 'DeviceProtection_Y', 'OnlineBackup_Y', 'OnlineSecurity_Y', 'StreamingMovies_Y', 
        'StreamingTV_Y', 'TechSupport_Y', 'No_Internet', 'DSL_Y', 'MultipleLines_Y', 'SeniorCitizen']

reg_lr = df_org[X_reg + ['MonthlyCharges']].drop_nulls().pipe(
    lambda x: LinearRegression(fit_intercept = True).fit(x[X_reg], x['MonthlyCharges'])
)
reg_lgb = df_org[X_reg + ['MonthlyCharges']].drop_nulls().pipe(
    lambda x: lgb.LGBMRegressor(verbose=-1).fit(x[X_reg], x['MonthlyCharges'])
)
df_train = df_train.with_columns(
    MonthlyCharges_exp_lr = reg_lr.predict(df_train[X_reg]),
    MonthlyCharges_exp_lgb = reg_lgb.predict(df_train[X_reg])
).with_columns(
    TotalCharges_exp_lr = pl.col('MonthlyCharges_exp_lr') * pl.col('tenure'),
    tenure_exp_lr = pl.col('TotalCharges') / (pl.col('MonthlyCharges_exp_lr') + 1),
    TotalCharges_exp_lgb = pl.col('MonthlyCharges_exp_lgb') * pl.col('tenure'),
    tenure_exp_lgb = pl.col('TotalCharges') / pl.col('MonthlyCharges_exp_lgb')
)
df_test = df_test.with_columns(
    MonthlyCharges_exp_lr = reg_lr.predict(df_test[X_reg]),
    MonthlyCharges_exp_lgb = reg_lgb.predict(df_test[X_reg])
).with_columns(
    TotalCharges_exp_lr = pl.col('MonthlyCharges_exp_lr') * pl.col('tenure'),
    tenure_exp_lr = pl.col('TotalCharges') / (pl.col('MonthlyCharges_exp_lr') + 1),
    TotalCharges_exp_lgb = pl.col('MonthlyCharges_exp_lgb') * pl.col('tenure'),
    tenure_exp_lgb = pl.col('TotalCharges') / pl.col('MonthlyCharges_exp_lgb')
)
X_exp = ['MonthlyCharges_exp_lr', 'MonthlyCharges_exp_lgb', 'TotalCharges_exp_lr', 'TotalCharges_exp_lgb', 'tenure_exp_lr', 'tenure_exp_lgb']
X_exp_lr = ['MonthlyCharges_exp_lr', 'TotalCharges_exp_lr', 'tenure_exp_lr']
X_exp_lgb = ['MonthlyCharges_exp_lgb', 'TotalCharges_exp_lgb', 'tenure_exp_lgb']

In [4]:
if os.path.exists('exp/skf5'):
    e_skf5 = Experimenter.load('exp/skf5', df_train)
    if e_skf5.status == 'closed':
        e_skf5.reopen_exp()
else:
    e_skf5 = Experimenter.create(
        df_train, 'exp/skf5', title='The experimentation using 5-fold StratifiedKFold',
        sp=StratifiedKFold(n_splits=5, shuffle=True, random_state=1),
        splitter_params={'y': target}
    )

e_skf5.set_grp('pre', role='stage', method='transform')
y_edges = {'y': [(None, target)]}
e_skf5.set_grp(
    'clf', role='head', method='predict_proba', edges=y_edges
)
e_skf5.add_collector(
    MetricCollector(
        'AUC', Connector(edges=y_edges, role='head'), slice(-1, None), roc_auc_score, include_train = True
    )
)
e_skf5.add_collector(
    StackingCollector(
        'stacking', Connector(edges=y_edges, role='head'),
        slice(-1, None), method='mean', experimenter = e_skf5
    )
)
e_skf5.set_grp('xgb', parent='clf', processor=xgb.XGBClassifier,
    adapter=XGBoostAdapter(eval_mode='valid'),
    params={
        'enable_categorical': True, 'device': 'cuda',
        'verbosity': 0, 'random_state': 1,
        'learning_rate': 0.05, 'colsample_bytree': 0.75, 'subsample': 0.9, 'reg_lambda': 0.01
    }
)

e_skf5.set_grp('lgb', parent='clf', processor=lgb.LGBMClassifier,
    adapter=LightGBMAdapter(eval_mode='valid'),
    params={
        'learning_rate': 0.05, 'verbose': -1, 'random_state': 1, 'min_samples_leaf': 512, 'colsample_bytree': 0.75, 
        'subsample': 0.75, 'subsample_freq': 1,'reg_lambda': 1
    }
)

e_skf5.set_grp('cb', parent='clf', processor=cb.CatBoostClassifier,
    adapter=CatBoostAdapter(eval_mode='valid'),
    params={
        'learning_rate': 0.05, 'verbose': 0, 'random_state': 1, 'max_depth': 5, 'colsample_bylevel': 0.75
    }
)

## Neural network
e_skf5.set_grp('nn', parent = 'clf', processor = NNClassifier, params = {'metrics': ['auc'], 'early_stopping': 0, 'validation_fraction': 0})

Markdown(
    e_skf5.desc_spec()
)

📁 Created directory: exp/skf5
Collect 5/5 (100%)        
Collect 5/5 (100%)        


## The experimentation using 5-fold StratifiedKFold

| 항목 | 값 |
|------|-----|
| **Outer Splitter (sp)** | `StratifiedKFold(n_splits=5, random_state=1, shuffle=True)` |
| **Inner Splitter (sp_v)** | None |
| **Splitter Params** | `{y='Churn'}` |
| **Outer Folds** | 5 |
| **Inner Folds** | 1 |

In [5]:
from sklearn.preprocessing import KBinsDiscretizer, RobustScaler, PolynomialFeatures, StandardScaler, TargetEncoder, MinMaxScaler
from mllabs.processor import FrequencyEncoder, CatConverter, CatOOVFilter, TypeConverter
from sklearn.impute import SimpleImputer
e_skf5.set_node(
    'si', grp='pre', processor=SimpleImputer, edges = {'X': [(None, X_num)]}
)
e_skf5.set_node(
    'kbin', grp='pre', processor=KBinsDiscretizer, edges = {'X': [('si', None)]}, params = {'n_bins': 10, 'strategy': 'uniform', 'encode': 'ordinal',  'random_state': 123}
)
e_skf5.set_node(
    'kbin_1', grp='pre', processor=KBinsDiscretizer, edges = {'X': [('si', ['.*TotalCharges', '.*MonthlyCharges'])]},
    params = {'n_bins': [4000, 200], 'strategy': 'uniform', 'encode': 'ordinal',  'random_state': 123}
)
e_skf5.set_node(
    'kbin_2', grp='pre', processor=KBinsDiscretizer, edges = {'X': [('si', ['.*TotalCharges', '.*MonthlyCharges'])]},
    params = {'n_bins': [500, 100], 'strategy': 'uniform', 'encode': 'ordinal',  'random_state': 123}
)
e_skf5.set_node(
    'rb', grp='pre', processor=RobustScaler, edges = {'X': [('si', None)]}, params = {}
)
e_skf5.set_node(
    'std', grp='pre', processor=StandardScaler, edges = {'X': [('si', None)]}, params = {}
)
e_skf5.set_node(
    'tgt_num', grp='pre', processor=TargetEncoder, method = 'fit_transform', edges = {'X': [('si', None)], **y_edges}, params = {'target_type': 'binary'}
)
e_skf5.set_node(
    'freq_num', grp='pre', processor=FrequencyEncoder, method = 'transform', edges = {'X': [('si', None)], **y_edges}, params = {'normalize': True}
)
e_skf5.set_node(
    'n2s', grp='pre', processor=TypeConverter, method = 'transform', edges = {'X': [('si', None)]}, params={'to': 'str'}
)
e_skf5.set_node(
    'n2c', grp='pre', processor=CatConverter, method = 'transform', edges = {'X': [('n2s', None), ('kbin', None), ('kbin_1', None), ('kbin_2', None)]}
)
e_skf5.set_node(
    'b2s', grp='pre', processor=TypeConverter, method = 'transform', edges = {'X': [(None, X_bin), (None, X_bin2)]}, params={'to':'str'}
)
e_skf5.set_node(
    'b2c', grp='pre', processor=CatConverter, method = 'transform', edges = {'X': [('b2s', None)]}
)

e_skf5.set_node(
    'coov', grp='pre', processor=CatOOVFilter, method = 'transform', edges = {'X': [('n2c', None)]}, params={'min_frequency': 10, 'missing_value': 'OOV'}
)

e_skf5.set_node(
    'rb_std', grp='pre', processor=RobustScaler, edges = {'X': [('rb', None)]}, params = {}
)
e_skf5.set_node(
    'expr4', grp='pre', processor = ExprProcessor, edges = {'X': [('si', None)]},
    params={'dict_expr': {
        'TotalCharges_per_tenure': pl.col('si__TotalCharges') / pl.col('si__tenure'),
        'Total_per_Monthly': pl.col('si__TotalCharges') / pl.col('si__MonthlyCharges'),
        'Monthly_per_Total': pl.col('si__MonthlyCharges') / pl.col('si__TotalCharges'),
        'Monthly_Avg_ratio': pl.col('si__MonthlyCharges') / (pl.col('si__TotalCharges') / pl.col('si__tenure')),
        'tenure_sq': pl.col('si__tenure') ** 2,
    }, 'with_columns': False}
)
service = ['DeviceProtection_Y', 'OnlineBackup_Y', 'OnlineSecurity_Y', 'StreamingMovies_Y', 'StreamingTV_Y', 'TechSupport_Y']
e_skf5.set_node(
    'expr5', grp='pre', processor = ExprProcessor, edges = {'X': [(None, service)]},
    params={'dict_expr': {
        'Service_cnt':  pl.sum_horizontal(service),
    }, 'with_columns': False}
)
e_skf5.set_node(
    'expr4_std', grp='pre', processor = StandardScaler, edges = {'X': [('expr4', None)]}
)
e_skf5.set_node(
    'exp_std', grp='pre', processor = StandardScaler, edges = {'X': [(None, X_exp)]}
)
e_skf5.build()

Building 18 node(s)
Build 5/5 (100%)                       
Build complete: 18 node(s)


In [6]:
from mllabs.processor import CatPairCombiner
from itertools import combinations

TOP_CATS_FOR_NGRAM = ['Contract', 'DSL_Y', 'PaymentMethod', 'OnlineSecurity']

bigram_cols = [(i1, i2) for i1, i2 in combinations(TOP_CATS_FOR_NGRAM, 2)]
trigram_cols = [(i1, i2, i3) for i1, i2, i3 in combinations(TOP_CATS_FOR_NGRAM, 3)]

e_skf5.set_node(
    'bigram', grp='pre', processor = CatPairCombiner, edges = {'X': [(None, TOP_CATS_FOR_NGRAM)]}, params={'pairs': bigram_cols}
)
e_skf5.set_node(
    'trigram', grp='pre', processor = CatPairCombiner, edges = {'X': [(None, TOP_CATS_FOR_NGRAM[:4])]}, params={'pairs': trigram_cols}
)
e_skf5.set_node(
    'com3', grp='pre', processor = CatPairCombiner, edges = {'X': [(None, ['Contract', 'InternetService', 'PaymentMethod'])]}, 
    params={'pairs': [('Contract', 'InternetService', 'PaymentMethod')]}
)
e_skf5.set_node(
    'tgt_bigram', grp='pre', processor=TargetEncoder, method = 'fit_transform', edges = {'X': [('bigram', None)], **y_edges}, params = {'target_type': 'binary'}
)
e_skf5.set_node(
    'tgt_trigram', grp='pre', processor=TargetEncoder, method = 'fit_transform', edges = {'X': [('trigram', None)], **y_edges}, params = {'target_type': 'binary'}
)
e_skf5.set_node(
    'coov_bigram', grp='pre', processor = CatOOVFilter, edges = {'X': [('bigram', None)]}, params={'min_frequency': 10, 'missing_value': 'OOV'}
)
e_skf5.set_node(
    'coov_trigram', grp='pre', processor = CatOOVFilter, edges = {'X': [('trigram', None)]}, params={'min_frequency': 10, 'missing_value': 'OOV'}
)
e_skf5.set_node(
    'exprd', grp='pre', processor = ExprProcessor, edges = {'X': [('si', None)]},
    params={'dict_expr': {
        'TotalCharges_3': ((pl.col('si__TotalCharges') * 1e-3).cast(pl.Int8) % 10).cast(pl.String).cast(pl.Categorical)
    }, 'with_columns': False}
)
e_skf5.build()

Building 8 node(s)
Build 5/5 (100%)                        
Build complete: 8 node(s)


In [7]:
i = 6
e_skf5.set_node(
    'lgb_{}'.format(i), grp = 'lgb', edges={
        'X': [(None, X_bin + X_nom + X_bin2 + X_num), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None), ('expr4', None)]
    }, params={'num_leaves': 7, 'n_estimators': 2500, 'learning_rate': 0.05}
)
e_skf5.exp()
e_skf5.get_collector('AUC').get_metrics_agg('lgb_{}'.format(i))[0].sort_values('valid', ascending = False)

Experimenting 1 node(s)
Exp 5/5 (100%)                                   
Experimentation complete: 1 node(s)


,valid,train_sub
lgb_6,0.918299,0.922702


In [7]:
i = 9
e_skf5.set_node(
    'lgb_{}'.format(i), grp = 'lgb', edges={
        'X': [(None, X_bin + X_nom + X_bin2 + X_num + X_exp), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None), ('expr4', None), ('expr5', None), ('tgt_bigram', None), ('tgt_trigram', None)]
    }, params={'num_leaves': 7, 'n_estimators': 3000, 'learning_rate': 0.05, 'colsample_bytree':0.5}
)
e_skf5.exp()
e_skf5.get_collector('AUC').get_metrics_agg('lgb_{}'.format(i))[0].sort_values('valid', ascending = False)

Experimenting 0 node(s)
Exp 5/5 (100%)        
Experimentation complete: 0 node(s)


,valid,train_sub
lgb_9,0.918368,0.923741


In [20]:
i = 10
e_skf5.set_node(
    'lgb_{}'.format(i), grp = 'lgb', edges={
        'X': [(None, X_bin + X_nom + X_bin2 + X_num + X_exp), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None), ('expr4', None), ('expr5', None), ('tgt_bigram', None), ('tgt_trigram', None)]
    }, params={'num_leaves': 31, 'n_estimators': 2000, 'learning_rate': 0.02, 'colsample_bytree':0.25}
)
e_skf5.exp()
e_skf5.get_collector('AUC').get_metrics_agg('lgb_{}'.format(i))[0].sort_values('valid', ascending = False)

Experimenting 1 node(s)
Exp 5/5 (100%)                                    
Experimentation complete: 1 node(s)


,valid,train_sub
lgb_10,0.918421,0.925692


In [21]:
i = 7
e_skf5.set_node(
    'xgb_{}'.format(i), grp = 'xgb', edges={
        'X': [(None, X_bin + X_nom + X_bin2 + X_num + X_exp), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None), ('expr4', None)]
    }, params={'max_depth': 3, 'n_estimators': 2500, 'learning_rate': 0.05}
)
e_skf5.exp()
e_skf5.get_collector('AUC').get_metrics_agg('xgb_{}'.format(i))[0].sort_values('valid', ascending = False)

Experimenting 1 node(s)
Exp 5/5 (100%)                                   
Experimentation complete: 1 node(s)


,valid,train_sub
xgb_7,0.918324,0.922757


In [31]:
i = 9
e_skf5.set_node(
    'xgb_{}'.format(i), grp = 'xgb', edges={
        'X': [(None, X_bin + X_nom + X_bin2 + X_num + X_exp), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None), ('expr4', None), ('expr5', None), ('tgt_bigram', None), ('tgt_trigram', None)]
    }, params={'max_depth': 4, 'n_estimators': 2500, 'learning_rate': 0.03, 'colsample_bytree': 0.35}
)
e_skf5.exp()
e_skf5.get_collector('AUC').get_metrics_agg('xgb_{}'.format(i))[0].sort_values('valid', ascending = False)

Experimenting 1 node(s)
Exp 5/5 (100%)                                   
Experimentation complete: 1 node(s)


,valid,train_sub
xgb_9,0.918407,0.924625


In [38]:
i = 5
e_skf5.set_node(
    'cb_{}'.format(i), grp = 'cb', edges={
        'X': [(None, X_bin + X_nom + X_bin2 + X_num + X_exp), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None), ('expr4', None), ('expr5', None), 
              ('tgt_bigram', None), ('tgt_trigram', None)]
    }, params={'colsample_bylevel':0.5, 'n_estimators': 3500, 'cat_features': ColSelector(col_type = 'category')}
)
e_skf5.exp()
e_skf5.get_collector('AUC').get_metrics_agg('cb_{}'.format(i))[0].sort_values('valid', ascending = False)

Experimenting 1 node(s)
Exp 5/5 (100%)                 
Experimentation complete: 1 node(s)


,valid,train_sub
cb_5,0.918285,0.926984


In [39]:
i = 6
e_skf5.set_node(
    'cb_{}'.format(i), grp = 'cb', edges={
        'X': [(None, X_nom + X_num + X_exp), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None), ('expr4', None), ('expr5', None), 
              ('tgt_bigram', None), ('tgt_trigram', None), ('b2c', None)]
    }, params={'colsample_bylevel':0.5, 'n_estimators': 3500, 'cat_features': ColSelector(col_type = 'category')}
)
e_skf5.exp()
e_skf5.get_collector('AUC').get_metrics_agg('cb_{}'.format(i))[0].sort_values('valid', ascending = False)

Experimenting 1 node(s)
Exp 5/5 (100%)                 
Experimentation complete: 1 node(s)


,valid,train_sub
cb_6,0.918253,0.92699


In [22]:
e_skf5.set_node(
    'nn_4', grp='nn', 
    edges={'X': [(None, X_bin + X_nom + X_bin2), ('std', None), ('expr4_std', None), ('exp_std', None), ('coov', None), ('exprd', None), 
                 ('coov_bigram', None), ('coov_trigram', None), ('tgt_num', None), ('freq_num', None)]},
    params={'hidden': {'units': [256, 128], 'dropout': 0.5}, 'cat_cols': ColSelector(col_type = 'category'), 'epochs': 4, 'learning_rate': 0.0001}
)
e_skf5.exp()

Experimenting 1 node(s)
Exp 0/5 (0%) > nn_4 0/1 (0%)

2026-03-18 07:12:16.107789: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


Exp 5/5 (100%)                                                      
Experimentation complete: 1 node(s)


In [23]:
e_skf5.get_collector('AUC').get_metrics_agg()[0].sort_values('valid', ascending = False)

,valid,train_sub
lgb_10,0.918421,0.925692
lgb_9,0.918378,0.923750
xgb_7,0.918324,0.922757
lgb_6,0.918321,0.922734
lgb_4,0.918104,0.922086
nn_4,0.917674,0.922280


# Stacking

In [40]:
df_stacking = e_skf5.get_collector('stacking').get_dataset()
df_stacking.head()

xgb_9__Churn_1,cb_6__Churn_1,lgb_9__Churn_1,cb_5__Churn_1,lgb_6__Churn_1,xgb_7__Churn_1,lgb_10__Churn_1,nn_4__Churn_1,Churn
f64,f64,f64,f64,f64,f64,f64,f64,i8
0.013242,0.009272,0.007996,0.008537,0.010924,0.010708,0.008031,0.010621,0
0.012123,0.013417,0.00954,0.015756,0.012998,0.008294,0.013918,0.009718,0
0.953979,0.960619,0.955376,0.962794,0.963181,0.956556,0.955962,0.939937,1
0.572572,0.553677,0.577041,0.558153,0.560555,0.567135,0.543794,0.575157,0
0.181884,0.200968,0.183675,0.198051,0.220498,0.214781,0.18191,0.131011,0


In [41]:
if os.path.exists('exp/stacking_aug'):
    e_stacking = Experimenter.load('exp/stacking_aug', df_stacking)
    if e_stacking.status == 'closed':
        e_stacking.reopen_exp()
else:
    e_stacking = Experimenter.create(
        df_stacking, 'exp/stacking_aug', title='Stacking',
        sp=StratifiedKFold(n_splits=5, random_state=1, shuffle=True),
        splitter_params={'y': target}
    )

e_stacking.add_collector(
    MetricCollector(
        'AUC',
        Connector(edges={'y': [(None, target)]}),
        slice(-1, None),
        roc_auc_score,
        include_train = True
    )
)

e_stacking.set_grp('pre', role='stage', method='transform')
e_stacking.build()
e_stacking.set_grp(
    'clf', role='head', method='predict_proba',
    edges={'y': [(None, target)]}
)
e_stacking.set_grp('lr', parent='clf', processor = LogisticRegression)

Loaded: 1 node(s), 3 group(s), 5 fold(s)
Building 0 node(s)
Build 5/5 (100%)        
Build complete: 0 node(s)


{'result': 'skip',
 'grp': <mllabs._pipeline.PipelineGroup at 0x7f6f932f5610>,
 'affected_nodes': []}

In [45]:
X_sel = ['{}__Churn_1'.format(i) for i in ['lgb_6', 'lgb_9', 'lgb_10', 'nn_4', 'xgb_7', 'xgb_9', 'cb_5', 'cb_6']]
e_stacking.set_node('lr20', grp = 'lr', edges = {'X': [(None, X_sel)]})
e_stacking.exp()
e_stacking.get_collector('AUC').get_metrics_agg()[0]

Experimenting 1 node(s)
Exp 5/5 (100%)                 
Experimentation complete: 1 node(s)


,valid,train_sub
lr20,0.918906,0.918907


In [46]:
from mllabs.collector import ModelAttrCollector
coef = ModelAttrCollector(
    'lr_coef', Connector(processor=LogisticRegression, edges=y_edges), 'coef'
)
e_stacking.collect(coef)
coef.get_attrs_agg('lr20')

Collect 5/5 (100%)                 


0  cb_5__Churn_1      0.513690
   cb_6__Churn_1      0.479243
   intercept         -3.318598
   lgb_10__Churn_1    0.579471
   lgb_6__Churn_1     0.494997
   lgb_9__Churn_1     0.378402
   nn_4__Churn_1      2.547735
   xgb_7__Churn_1     0.700712
   xgb_9__Churn_1     0.749131
dtype: float64

In [47]:
e_stacking.get_collector('AUC').get_metrics_agg()[0].sort_values('valid', ascending=False).iloc[:5]

,valid,train_sub
lr20,0.918906,0.918907


# Submission

In [48]:
e_skf5.add_trainer('trainer')
e_skf5.trainers['trainer'].select_head(['lgb_6', 'lgb_9', 'lgb_10', 'nn_4', 'xgb_7', 'xgb_9', 'cb_5', 'cb_6'])
e_skf5.trainers['trainer'].train()

nn_4 32/32 (100%)                                                       
Train complete: 32 node(s)


In [49]:
e_stacking.add_trainer('trainer')
e_stacking.trainers['trainer'].select_head(['lr20'])
e_stacking.trainers['trainer'].train()

lr20 1/1 (100%)                 
Train complete: 1 node(s)


In [55]:
from mllabs.collector import ModelAttrCollector
coef = ModelAttrCollector('lr_coef', Connector(processor=LogisticRegression, edges=y_edges), 'coef')
e_stacking.collect(coef)
coef.get_attrs_agg('lr20')

Collect 5/5 (100%)                 


0  cb_5__Churn_1      0.513690
   cb_6__Churn_1      0.479243
   intercept         -3.318598
   lgb_10__Churn_1    0.579471
   lgb_6__Churn_1     0.494997
   lgb_9__Churn_1     0.378402
   nn_4__Churn_1      2.547735
   xgb_7__Churn_1     0.700712
   xgb_9__Churn_1     0.749131
dtype: float64

In [58]:
df_result = e_stacking.trainers['trainer'].to_inferencer(slice(-1, None)).process(
    e_skf5.trainers['trainer'].to_inferencer(slice(-1, None)).process(df_test)
)[['lr20__Churn_1']]
df_result.columns = ['Churn']

In [59]:
df_result.with_columns(
    df_test['id']
)[['id', 'Churn']].write_csv('data/submission.csv')

In [58]:
# !kaggle competitions submit -c playground-series-s6e3 -f data/submission.csv -m "10"

100%|██████████████████████████████████████| 6.51M/6.51M [00:05<00:00, 1.26MB/s]
Successfully submitted to Predict Customer Churn